# EMA Touch-and-Rejection

EMA touch-and-rejection — a pullback strategy ported from the standalone `ema` project. It waits for price to *pull back into* an EMA and *reject* it: a bar whose wick tags the EMA (within a `delta` tolerance) but which closes back on the trend side. Trades both directions; long is tried first on an ambiguous bar and the run's direction gate decides which sides are allowed.

__How the EMA Touch Algorithm Determines Entry/Exit:__
- One entry EMA (`ema_touch_period`, default 50); an optional slower regime EMA can gate each side.
- Long Entry: the bar's **low** comes within `delta` of the EMA AND the bar **closes >= EMA** (rejection up). With a regime filter: only if close > the regime EMA.
- Short Entry: the bar's **high** comes within `delta` of the EMA AND the bar **closes <= EMA** (rejection down).
- `delta` units follow `ema_touch_delta_mode`: `absolute` (quote points), `percent` (% of the EMA), or `atr` (× ATR, cross-symbol).
- Exit: the injected exit policy — default a 1% fixed stop + a 3R take-profit, stop-first on an ambiguous bar (both checked intrabar).
- An entry whose stop would land on the wrong side of the fill is skipped.

## Configuration: automatic

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import EmaTouchParams
from engine.visualization import build_chart

In [ ]:
# Automatic config: canonical handles from the three configurators (project-wide defaults).
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
import dataclasses
from engine.data_configurator import ACTIVE, load_data, save_result
from engine.strategy_configurator import EmaTouchParams, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE

DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = EmaTouchParams()    # engine/strategy_configurator.py (signal knobs)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

## Configuration: manual

Per-notebook overrides on top of the automatic config above. Each cell applies
`dataclasses.replace` to one handle; **leave a dict empty (or `EXIT_POLICY = None`)
to keep that dimension automatic**. Everything below this chapter uses only
`df`, `SYMBOL`, `INTERVAL`, `STRATEGY_CONFIG`, `EXIT_POLICY`, `TRADING_CONFIG`.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic EmaTouchParams().
STRATEGY_OVERRIDES = {}      # e.g. {"ema_touch_period": 34, "ema_touch_delta_mode": "atr"}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

In [ ]:
# Resolve the final inputs the rest of the notebook uses. (Runs after overrides.)
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

## EMA Touch-and-Rejection

In [ ]:
# Import EMA touch-and-rejection strategy
from engine.strategies import EmaTouchStrategy

In [ ]:
# Backtest EMA touch-and-rejection strategy.
# Key signal knobs (EmaTouchParams): ema_touch_period (EMA span), ema_touch_delta
# + ema_touch_delta_mode ('absolute' quote-points | 'percent' | 'atr'), and the
# optional ema_touch_regime_filter. The 40-point absolute default suits BTCUSDT;
# switch to delta_mode='atr' for cross-symbol use. Stop/TP come from the exit
# policy (default 1% stop + 3R target).
strategy = EmaTouchStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# EMA touch-and-rejection strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()